In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Import PyTorch packages
import torch
import torch.nn as nn

In [ ]:
# Check the version of PyTorch
print(torch.__version__)

In [ ]:
if not torch.cuda.is_available():
    raise SystemError(
        "No GPU found!\n"
        "To enable GPU, click on top menu: Runtime → Change runtime type → Hardware Accelerator → T4 GPU → Save"
    )

device = torch.device('cuda')
print(f'Using device: {torch.cuda.get_device_name(0)}')

In [ ]:
# Access to Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Load Raw Data and Extract Acceleration Data
- Generate single array that consists of every acceleration data (normal and abnormal)

In [ ]:
NoOfData = 180
Normal   = []
Abnormal = []

for i in range(NoOfData):
    temp_path1 = f'https://github.com/purduelamm/purdue_me597_iiot/blob/main/ml_tutorial/Dataset/Normal_{i+1}?raw=true'
    temp_path2 = f'https://github.com/purduelamm/purdue_me597_iiot/blob/main/ml_tutorial/Dataset/Abnormal_{i+1}?raw=true'

    Normal.append(pd.read_csv(temp_path1,   sep=',', header=None))
    Abnormal.append(pd.read_csv(temp_path2, sep=',', header=None))

In [ ]:
print(Normal[0].shape, Abnormal[0].shape)

In [ ]:
DataLength = len(Normal[0])

AccData_Nor = pd.DataFrame(np.zeros((NoOfData, DataLength)))
AccData_Abn = pd.DataFrame(np.zeros((NoOfData, DataLength)))

for i in range(NoOfData):
    AccData_Nor.iloc[i,:] = Normal[i].iloc[:,1]
    AccData_Abn.iloc[i,:] = Abnormal[i].iloc[:,1]

AccData = np.array(pd.concat([AccData_Nor, AccData_Abn], axis=0))
AccData.shape

# Convert Acceleration Data into Spectrogram by STFT

[Tip]

You can define the size of spectrogram (resolution of time and frequency)

by adjusting 'Number of samples(N) per segment (nperseg)' and 'Number of samples(N) for overlap'

In [ ]:
from scipy import signal

Fs = 12800  # Sampling Frequency
f,t,AccSTFT = signal.spectrogram(AccData, Fs, nperseg = 78, noverlap = 10)
AccSTFT.shape

Compare spectrograms between normal and abnormal

In [ ]:
idx = 1  # Select index (1~180)

plt.figure(figsize=(12,4))

plt.subplot(1,2,1)
plt.pcolormesh(t, f, AccSTFT[idx-1], cmap='jet')
plt.title(f"STFT (Normal_{idx})", fontsize=15)
plt.xlabel('Time(s)', fontsize=12)
plt.ylabel('Frequency(Hz)', fontsize=12)
plt.colorbar()

plt.subplot(1,2,2)
plt.pcolormesh(t, f, AccSTFT[idx+NoOfData-1], cmap='jet')
plt.title(f"STFT (Abnormal_{idx})", fontsize=15)
plt.xlabel('Time(s)', fontsize=12)
plt.colorbar()

plt.show()

## Split Training & Test Data
- Use 'train_test_split' function
- It randomly samples the training and testing data according to the designated ratio.

In [ ]:
# Split Training & Test Data
NormalSet   = AccSTFT[:NoOfData]
AbnormalSet = AccSTFT[NoOfData:]

NoOfSensor  = 1
NormalSet   = NormalSet.reshape(NormalSet.shape[0], NormalSet.shape[1], NormalSet.shape[2], NoOfSensor)
AbnormalSet = AbnormalSet.reshape(AbnormalSet.shape[0], AbnormalSet.shape[1], AbnormalSet.shape[2], NoOfSensor)

print(NormalSet.shape, AbnormalSet.shape)

In [ ]:
# Split while still NumPy
from sklearn.model_selection import train_test_split
TestData_Ratio = 0.2

TrainData_Nor, TestData_Nor = train_test_split(NormalSet,   test_size=TestData_Ratio, random_state=777)
TrainData_Abn, TestData_Abn = train_test_split(AbnormalSet, test_size=TestData_Ratio, random_state=777)

# Convert to PyTorch AND swap channels from last to first in one step
TrainData_Nor = torch.tensor(TrainData_Nor, dtype=torch.float32).permute(0,3,1,2).to(device)
TestData_Nor  = torch.tensor(TestData_Nor,  dtype=torch.float32).permute(0,3,1,2).to(device)
TrainData_Abn = torch.tensor(TrainData_Abn, dtype=torch.float32).permute(0,3,1,2).to(device)
TestData_Abn  = torch.tensor(TestData_Abn,  dtype=torch.float32).permute(0,3,1,2).to(device)

print(TrainData_Nor.shape, TestData_Nor.shape)

## Data Labling (class index encoding)

In [ ]:
# Normal = 0, Abnormal = 1
TrainLabel_Nor = torch.zeros(TrainData_Nor.shape[0], dtype=torch.long).to(device)
TrainLabel_Abn = torch.ones( TrainData_Abn.shape[0], dtype=torch.long).to(device)
TestLabel_Nor  = torch.zeros(TestData_Nor.shape[0],  dtype=torch.long).to(device)
TestLabel_Abn  = torch.ones( TestData_Abn.shape[0],  dtype=torch.long).to(device)

print(TrainLabel_Nor.shape, TestLabel_Nor.shape)
print(TrainLabel_Abn.shape, TestLabel_Abn.shape)

## Data and Label Preparation

In [ ]:
TrainData  = torch.cat([TrainData_Nor,  TrainData_Abn ], dim=0)
TestData   = torch.cat([TestData_Nor,   TestData_Abn  ], dim=0)
TrainLabel = torch.cat([TrainLabel_Nor, TrainLabel_Abn], dim=0)
TestLabel  = torch.cat([TestLabel_Nor,  TestLabel_Abn ], dim=0)

print(TrainData.shape,  TestData.shape)
print(TrainLabel.shape, TestLabel.shape)

## Setting hyperparameters for training CNN(Convolutional Neural Network)

In [ ]:
learningRate  = 0.0005
Epoch         = 1000

## Designing an CNN architecture

In [ ]:
class CNN_model(nn.Module):
    def __init__(self, input_shape):
        super(CNN_model, self).__init__()

        # Convolutional layers
        self.conv1    = nn.Conv2d(in_channels=input_shape[0], out_channels=16,  kernel_size=3, stride=1, padding=1)  # Convolution layer 1
        self.pool1    = nn.MaxPool2d(kernel_size=2, stride=2)                                                        # Pooling layer 1
        self.conv2    = nn.Conv2d(in_channels=16,               out_channels=32,  kernel_size=3, stride=1, padding=1)  # Convolution layer 2
        self.pool2    = nn.MaxPool2d(kernel_size=2, stride=2)                                                        # Pooling layer 2

        self.relu     = nn.ReLU()
        self.flatten  = nn.Flatten()                                                                                  # Flatten layer

        # Correct - must use * 32 (conv2 now has 32 filters)
        self.flat_size = (input_shape[1]//4) * (input_shape[2]//4) * 32

        self.dense    = nn.Linear(self.flat_size, 64)                                                                 # Dense layer
        self.output   = nn.Linear(64, 2)                                                                              # Output layer

    def forward(self, x):
        x = self.pool1(self.relu(self.conv1(x)))  # Conv1 → ReLU → Pool1
        x = self.pool2(self.relu(self.conv2(x)))  # Conv2 → ReLU → Pool2
        x = self.flatten(x)                        # Flatten
        x = self.relu(self.dense(x))               # Dense → ReLU
        x = self.output(x)                         # Output
        return x

# Initialize model, loss function and optimizer
CnnModel  = CNN_model(TrainData.shape[1:]).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(CnnModel.parameters(), lr=learningRate)

In [ ]:
# Print model summary
print(CnnModel)

# Print number of parameters
total_params     = sum(p.numel() for p in CnnModel.parameters())
trainable_params = sum(p.numel() for p in CnnModel.parameters() if p.requires_grad)
print(f'\nTotal parameters:     {total_params}')
print(f'Trainable parameters: {trainable_params}')

# Print parameter shapes
for name, param in CnnModel.named_parameters():
    print(f'{name}: {param.shape}')

## CNN Model Training

In [ ]:
torch.manual_seed(777)

CnnModel.train()
TrainingHistory = {'loss': [], 'accuracy': []}

idx        = torch.randperm(TrainData.shape[0])
TrainData  = TrainData[idx]
TrainLabel = TrainLabel[idx]

for epoch in range(Epoch):
    optimizer.zero_grad()
    outputs  = CnnModel(TrainData)
    loss     = criterion(outputs, TrainLabel)
    loss.backward()
    optimizer.step()

    predicted = torch.argmax(outputs, dim=1)
    accuracy  = (predicted == TrainLabel).float().mean().item()

    TrainingHistory['loss'].append(loss.item())
    TrainingHistory['accuracy'].append(accuracy)

    if (epoch+1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{Epoch}] Loss: {loss.item():.4f}  Accuracy: {accuracy:.4f}')

In [ ]:
# Evaluation result for test data (not trained)
CnnModel.eval()
with torch.no_grad():
    outputs   = CnnModel(TestData)
    loss      = criterion(outputs, TestLabel)
    predicted = torch.argmax(outputs, dim=1)
    accuracy  = (predicted == TestLabel).float().mean().item()

print(f'Loss: {loss.item():.4f}  Accuracy: {accuracy:.4f}')
print('The closer the Loss is to 0 and the closer the accuracy is to 1 (100%), the better.')

In [ ]:
# Check the training process (Loss, Accuracy)

fig, loss_ax = plt.subplots(figsize=(8,6))
acc_ax = loss_ax.twinx()

loss_ax.plot(TrainingHistory['loss'],     label='train loss', c='tab:red')
loss_ax.set_xlabel('epoch', fontsize=15)
loss_ax.set_ylabel('loss',  fontsize=15)
loss_ax.legend(loc='center left', fontsize=12)

acc_ax.plot(TrainingHistory['accuracy'], label='train acc', c='tab:blue')
acc_ax.set_ylabel('accuracy', fontsize=15)
acc_ax.legend(loc='center right', fontsize=12)

plt.show()

Save ML model (CNN) as a file

In [ ]:
import os
os.makedirs('/content/drive/MyDrive/Colab Notebooks/SavedFiles/ML_Models', exist_ok=True)

torch.save(CnnModel.state_dict(), '/content/drive/MyDrive/Colab Notebooks/SavedFiles/ML_Models/CNN_model.pth')

Load the saved ML model (CNN) and test

In [ ]:
# Load the saved model
LoadedModel = CNN_model(TrainData.shape[1:]).to(device)
LoadedModel.load_state_dict(torch.load('/content/drive/MyDrive/Colab Notebooks/SavedFiles/ML_Models/CNN_model.pth'))

# Evaluate
LoadedModel.eval()
with torch.no_grad():
    outputs   = LoadedModel(TestData)
    loss      = criterion(outputs, TestLabel)
    predicted = torch.argmax(outputs, dim=1)
    accuracy  = (predicted == TestLabel).float().mean().item()

print('[Performance of CNN model] \n')
print('Accuracy : {:.2f}%'.format(accuracy*100))

In [ ]:
# Predicted result
LoadedModel.eval()
with torch.no_grad():
    Predicted = LoadedModel(TestData)
    Predicted = torch.softmax(Predicted, dim=1)

# Convert to numpy for sklearn
TestLabel_rev = TestLabel.cpu().numpy()
Predicted_rev = torch.argmax(Predicted, dim=1).cpu().numpy()

# Plot the confusion matrix
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(TestLabel_rev, Predicted_rev)

plt.figure(figsize=(6, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap=plt.cm.Blues, cbar=False, square=True)
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("Confusion Matrix of the CNN Model")
plt.show()

from sklearn import metrics

# Calculate the evaluation metrics
accuracy  = metrics.accuracy_score(TestLabel_rev, Predicted_rev)
precision = metrics.precision_score(TestLabel_rev, Predicted_rev)
recall    = metrics.recall_score(TestLabel_rev, Predicted_rev)
f1_score  = metrics.f1_score(TestLabel_rev, Predicted_rev)

print("\n\n")
print(f"CNN Model Evaluation:\n")
print(f"Accuracy : {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall   : {recall:.2f}")
print(f"F1 Score : {f1_score:.2f}")